# Human-in-the-Loop: Puertas previas a la acción, clasificación por riesgo y registro de auditoría

El README de esta lección introduce Human-in-the-Loop con un pequeño fragmento que pide al usuario `APPROVE` o `REJECT` después de que el agente ya ha producido una respuesta. Ese patrón es un buen punto de partida, pero las implementaciones de HITL en producción comúnmente necesitan tres elementos adicionales:

1. Una **puerta previa a la acción** que se ejecute **antes** de que el agente realice un paso riesgoso, para que el costo, la irreversibilidad y la latencia se mantengan bajo control.
2. **Clasificación por niveles de riesgo**, para que las acciones de bajo riesgo se ejecuten automáticamente, las acciones de riesgo medio se aprueben en lote, y solo las acciones de alto riesgo requieran la intervención humana.
3. Un **registro de auditoría más un ciclo de revisión**, para que cada decisión de la puerta se registre como JSONL, y un rechazo vuelva a solicitar al agente con una razón estructurada en lugar de solo imprimir `Revising...`.

Este cuaderno construye cada uno de estos sobre los mismos primitivos que `06-system-message-framework.ipynb`. Funciona de principio a fin en `DEMO_MODE = True` (no se necesita entrada interactiva) o con avisos reales `input()` cuando `DEMO_MODE = False`. Nota: en DEMO_MODE la reintención de la tercera meta está guionada para que la mecánica del ciclo sea visible de principio a fin. La reclasificación real basada en revisión requiere `DEMO_MODE = False` y un operador.

**Fuera de alcance (manejado en otras lecciones):** autenticación y control de acceso (Amenaza 2 del README de la Lección 06), middleware para llamada a herramientas (profundización en MAF de la Lección 14), patrones de debate multi-agente.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Patrón 1: Puerta previa a la acción

El fragmento HITL del README llama primero al agente y luego pide al usuario que apruebe el resultado. Ese es un flujo de **post-acción**. El agente ya ha ejecutado, por lo que el costo de la llamada LLM ya se pagó, y cualquier efecto secundario (correo enviado, fila de base de datos escrita, comentario publicado) ya ocurrió.

Un flujo de **pre-acción** inserta la puerta antes de que el agente ejecute el paso riesgoso. El agente propone la acción, la puerta decide si ejecutarla, y solo con la aprobación ocurre el efecto secundario.

| Aspecto | Aprobación post-acción (fragmento README) | Puerta pre-acción (este cuaderno) |
|---|---|---|
| ¿Cuándo se ejecuta la aprobación? | Después de que el agente produjo resultado | Antes de que se ejecute cualquier efecto secundario |
| Costo LLM en rechazo | Ya pagado | Se paga solo por la propuesta, no por la acción |
| Efectos secundarios irreversibles | Posibles (la acción ya ocurrió) | Prevenidos |
| Claridad en auditoría | La aprobación es una impresión | La aprobación es un registro JSON con marca de tiempo, acción, razón |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Patrón 2: Clasificación de riesgos

No todas las acciones necesitan aprobación humana. Una consulta de solo lectura contra una API pública tiene diferentes implicaciones que enviar un correo electrónico a un cliente. Tratar ambas igual desperdicia la atención del operador y ralentiza al agente.

Un modelo simple de 3 niveles:

| Nivel | Ejemplos | Flujo de aprobación |
|---|---|---|
| `bajo` (solo lectura) | Buscar en una base de conocimientos, consultar opciones de vuelo, obtener una página web pública | Autoejecución, registrado para auditoría |
| `medio` (mutación barata) | Cachear un resultado, incrementar un contador, programar un recordatorio | Autoejecución, pero revisión diaria agrupada |
| `alto` (de cara al exterior o irreversible) | Enviar un correo electrónico, cobrar una tarjeta, publicar en un canal público | Bloqueo para aprobación humana |

Esta es una forma de clasificación. Los sistemas en producción a menudo usan niveles más granulares (por ejemplo, niveles de permiso AWS IAM, niveles de acceso basados en roles). La versión de 3 niveles a continuación es la versión más pequeña útil para un agente que mezcla acciones de solo lectura y con efectos secundarios.

El clasificador a continuación usa heurísticas basadas en palabras clave para que la demostración sea determinista y económica. En un sistema de producción, intercambiarías esto por un clasificador aprendido o un motor de políticas.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Patrón 3: Registro de auditoría y bucle de revisión

Un `print("Response approved.")` no es un registro de auditoría. Para generar confianza, cada decisión del control debe registrarse como un evento estructurado que luego puedas consultar, reproducir o adjuntar a una revisión de incidentes.

Dos elementos:

1. **JSONL solo para anexar.** Una línea por decisión, con marca de tiempo, acción, nivel, decisión, razón. Fácil de buscar con grep, fácil de enviar a una verdadera tienda de registros más adelante.  
2. **Bucle de revisión en caso de rechazo.** Cuando el control devuelve `deny`, el agente se vuelve a solicitar a sí mismo con la razón del rechazo en contexto, para que la siguiente propuesta pueda evitar el problema.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Recursos adicionales

Varios otros proyectos públicos implementan variaciones de estos patrones HITL. Compare enfoques para encontrar lo que se ajusta a su stack:

- **LangChain** envoltorios de herramientas human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): envoltorios de herramientas plug-and-play que pausan la ejecución para la entrada humana.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ reestructuró esto): utiliza un rol de agente específicamente para representar al humano en conversaciones multi-agente.
- **Semantic Kernel** filtros de funciones ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware que se ejecuta alrededor de cada llamada a función, adecuado para lógica de control.

Cada proyecto maneja los tres subpatrones de manera diferente: LangChain los envuelve como herramientas, AutoGen usa un rol de agente, Semantic Kernel usa filtros middleware. Lea una o dos implementaciones completas antes de elegir un diseño para su propio agente.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
